# **HydroServer Example 4: Automated GEOGLOWS Streamflow Forecast Ingestion with HydroServer ETL Tasks**

### **Overview**

This exercise demonstrates how to configure HydroServer's **native ETL orchestration system** to automatically retrieve streamflow forecasts from the **GEOGLOWS River Forecast System (RFS)** and load them into a HydroServer Datastream.

The example uses GEOGLOWS V2 river reach **`160193736`**, located near gauge **1DA02** on the Nzoia River near Webuye, Kenya.

The workflow will:

1. Connect to HydroServer and reuse an existing Workspace.
2. Create the HydroServer resources required for the forecast Datastream.
3. Create a reusable **Data Connection** to the GEOGLOWS API.
4. Create an **ETL Task** that maps `flow_median` to the forecast Datastream.
5. Schedule the ETL Task to run automatically once per day.
6. Optionally trigger the task manually to test the configuration.

> **Important:** Automated ETL Tasks require HydroServer's orchestration system to be installed and running. The public HydroServer Playground can be used to review/configure the workflow, but scheduled ETL Tasks do not execute automatically there.


### **Import Required Python Packages**

Install and import the packages required to connect to HydroServer and configure the automated ETL workflow.


In [1]:
%pip install -q hydroserverpy==1.11.3

from getpass import getpass
import time
from hydroserverpy import HydroServer


Note: you may need to restart the kernel to use updated packages.


### **HydroServer-Native ETL Orchestration**

In this version of the exercise, we do **not** manually build an `ETLPipeline` with an Extractor, Transformer, and Loader.

Instead, HydroServer stores the source configuration as a **Data Connection** and the mapping/schedule as an **ETL Task**. When orchestration is enabled, HydroServer's worker executes the task automatically according to its schedule.


### **Connect to HydroServer**

Set the HydroServer URL and enter your credentials.

For a self-hosted HydroServer with orchestration enabled, replace the Playground URL with your HydroServer instance URL.


In [6]:
# Set the HydroServer URL
hydroserver_host = 'https://playground.hydroserver.org'

# Enter the email associated with your HydroServer account
hydroserver_email = 'svicario@lincolninst.edu'

# Enter your HydroServer password securely
hydroserver_password = getpass('Enter your HydroServer password: ')


Enter your HydroServer password:  ········


### **Initialize the HydroServer Connection**

Use your credentials to initialize the connection to the **HydroServer Playground**.

This connection will allow the notebook to create and manage HydroServer resources and later upload the GEOGLOWS streamflow forecast data.

In [7]:
# Initialize HydroServer connection with credentials.
hs = HydroServer(
    host=hydroserver_host,
    email=hydroserver_email,
    password=hydroserver_password
)

print('\nSuccessfully connected to HydroServer!')


Successfully connected to HydroServer!


### **Connect to Your Existing Workspace**

In the previous exercises, you created a HydroServer Workspace for the training. In this exercise, we will **reuse that existing Workspace** rather than create a new one.

Specify the name of your existing Workspace and retrieve its information from HydroServer. The Workspace ID will be used to create the resources needed for the GEOGLOWS forecast workflow.

In [8]:
# Enter the name of the workspace created in the previous exercise
workspace_name = "Kenya Training 2026"

# Retrieve workspaces associated with your account
workspaces = hs.workspaces.list(is_associated=True)

# Find the workspace by name
workspace = next(
    ws for ws in workspaces.items
    if ws.name == workspace_name
)

# Save the Workspace ID for later use
workspace_id = workspace.uid

print(f"Using workspace: {workspace.name}")
print(f"Workspace ID: {workspace_id}")

Using workspace: Kenya Training 2026
Workspace ID: 01a056f6-8571-71ed-b364-c6dbbb8ab1ec


### **Create the Nzoia River Monitoring Site**

Next, create a **monitoring site (Thing)** in HydroServer representing the Nzoia River near **Webuye, Kenya**.

For this exercise, we use the location of gauge **1DA02**. Its coordinates are also used to identify the nearest GEOGLOWS modeled river reach, **River ID `160193736`**, from which we will retrieve the streamflow forecasts.

The Monitoring Site ID is saved because it will be needed when creating the forecast Datastream.

In [9]:
# Create the monitoring site for the Nzoia River near Webuye, Kenya
# Gauge/reference station: 1DA02
# Coordinates used to identify the nearest GEOGLOWS river reach
nzoia_station = hs.things.create(
    workspace=workspace_id,
    name="Nzoia River near Webuye",
    description=(
        "Reference location for gauge 1DA02 on the Nzoia River near Webuye, Kenya. "
        "The nearest GEOGLOWS V2 modeled river reach is 160193736."
    ),
    sampling_feature_type="Site",
    sampling_feature_code="1DA02",
    site_type="Stream",
    latitude=0.58595,
    longitude=34.80685,
    country="KE",
    is_private=False
)

thing_id = nzoia_station.uid

print("Created monitoring site:")
print(f"{nzoia_station.name}: {thing_id}")


Created monitoring site:
Nzoia River near Webuye: 01a05721-8746-75a6-b17e-5ee394793bbb


### **Create the GEOGLOWS Model Metadata**

Next, create a **Sensor** in HydroServer to describe the source of the forecast data.

In this exercise, the Sensor represents the **GEOGLOWS River Forecast System (RFS) V2**, which provides the modeled streamflow forecasts that will be retrieved through the GEOGLOWS API.

The Sensor ID will later be associated with the forecast Datastream.

In [10]:
# Create metadata describing GEOGLOWS as the source/model
geoglows_sensor = hs.sensors.create(
    workspace=workspace_id,
    name="GEOGLOWS RFS V2",
    description=(
        "Modeled streamflow forecasts from the GEOGLOWS River Forecast System V2 "
        "for river reach 160193736."
    ),
    encoding_type="text/csv",
    method_type="Model Simulation",
    method_code="geoglows-rfs-v2"
)

print("Created sensor/model metadata:")
print(f"{geoglows_sensor.name}: {geoglows_sensor.uid}")


Created sensor/model metadata:
GEOGLOWS RFS V2: 01a05721-8e53-7931-a0e4-8928b6d07cc1


### **Create the Streamflow Observed Property**

Next, create the **Observed Property** that defines the variable represented by the forecast data.

For this exercise, the Observed Property is **Streamflow**, representing the forecasted river discharge provided by GEOGLOWS.

The Observed Property ID will later be associated with the forecast Datastream.

In [11]:
# Create the observed property
streamflow = hs.observedproperties.create(
    workspace=workspace_id,
    name="Streamflow",
    definition="Streamflow",
    description="Forecasted river discharge (streamflow).",
    observed_property_type="Hydrology",
    code="Streamflow"
)

print("Created observed property:")
print(f"{streamflow.name}: {streamflow.uid}")


Created observed property:
Streamflow: 01a05721-92b8-7c81-af22-74b1f820c60e


### **Create the Streamflow Unit**

Next, create the **Unit** associated with the Streamflow Observed Property.

GEOGLOWS reports streamflow forecasts in **cubic meters per second (m³/s)**. This unit will later be associated with the forecast Datastream.

In [12]:
# GEOGLOWS streamflow is reported in cubic meters per second
streamflow_unit = hs.units.create(
    workspace=workspace_id,
    name="Cubic meter per second",
    symbol="m^3/s",
    definition="Cubic meters per second",
    unit_type="Volumetric Flow Rate"
)

print("Created unit:")
print(f"{streamflow_unit.name}: {streamflow_unit.uid}")


Created unit:
Cubic meter per second: 01a05721-9409-70b1-826d-866d3afd7159


### **Create the Forecast Processing Level**

Next, create a **Processing Level** to indicate that the streamflow values are **forecasted model outputs** rather than observed measurements.

For this exercise, the Processing Level identifies the data as streamflow forecasts produced by **GEOGLOWS RFS V2**. It will later be associated with the forecast Datastream.

In [13]:
# Processing level for forecast/model output
forecast_processing = hs.processinglevels.create(
    workspace=workspace_id,
    code="Forecast",
    definition="Forecast streamflow",
    explanation="Streamflow forecast produced by GEOGLOWS RFS V2."
)

print("Created processing level:")
print(f"{forecast_processing.code}: {forecast_processing.uid}")


Created processing level:
Forecast: 01a05721-987b-7fa0-acc6-0da0909c385f


### **Create the GEOGLOWS Forecast Datastream**

Next, create the **Datastream** that will store the GEOGLOWS streamflow forecast in HydroServer.

The Datastream connects the **Nzoia River monitoring site** with the GEOGLOWS model, Streamflow Observed Property, unit, and Forecast Processing Level created in the previous steps.

Because GEOGLOWS provides forecasted streamflow at **3-hour intervals**, the Datastream is configured with a 3-hour time spacing and a forecast period extending up to **15 days ahead**.

In [14]:
forecast_datastream = hs.datastreams.create(
    name=f"GEOGLOWS Streamflow Forecast - {nzoia_station.name}",
    description=(
        "Latest GEOGLOWS RFS V2 streamflow forecast for river reach 160193736, "
        "near gauge 1DA02 on the Nzoia River."
    ),
    thing=nzoia_station.uid,
    sensor=geoglows_sensor.uid,
    observed_property=streamflow.uid,
    processing_level=forecast_processing.uid,
    unit=streamflow_unit.uid,
    observation_type="Model Simulation",
    result_type="Timeseries",
    sampled_medium="Surface Water",
    no_data_value=-9999,
    aggregation_statistic="Average",
    time_aggregation_interval=3,
    time_aggregation_interval_unit="hours",
    intended_time_spacing=3,
    intended_time_spacing_unit="hours",
    status="Ongoing",
    is_private=False,
    is_visible=True
)

### **Create the GEOGLOWS Data Connection**

A **Data Connection** tells HydroServer where the source data are located and how to read them.

For this exercise, the source is the GEOGLOWS V2 forecast API. The river reach is defined as a `per_task` placeholder, making the same Data Connection reusable for other GEOGLOWS river reaches.

The API returns a CSV file in which the `datetime` column contains forecast timestamps.


In [15]:
# Create a reusable Data Connection to the GEOGLOWS forecast API
geoglows_data_connection = hs.dataconnections.create(
    name="GEOGLOWS Streamflow Forecasts",
    workspace=workspace_id,
    source_url=(
        "https://geoglows.ecmwf.int/api/v2/"
        "forecast/{river_id}?format=csv"
    ),
    payload_type="CSV",
    timestamp_key="datetime",
    header_row=1,
    data_start_row=2,
    delimiter=",",
    placeholder_variables=[
        {
            "name": "river_id",
            "variable_type": "per_task"
        }
    ],
)

print(f"{geoglows_data_connection.name}: {geoglows_data_connection.uid}")


GEOGLOWS Streamflow Forecasts: 01a05721-ac3c-76da-a0f7-03af3ebbf93b


### **Create and Schedule the GEOGLOWS ETL Task**

Next, create an **ETL Task** that connects the GEOGLOWS Data Connection to the HydroServer forecast Datastream.

The task supplies River ID **`160193736`** and maps the GEOGLOWS `flow_median` series to the target Datastream.

The task is configured to run **once per day** when HydroServer orchestration is enabled.


In [16]:
# Create the scheduled ETL task
geoglows_etl_task = hs.etltasks.create(
    name="Daily Nzoia GEOGLOWS Streamflow Forecast",
    data_connection=geoglows_data_connection,
    task_variables={
        "river_id": "160193736"
    },
    mappings=[
        {
            "source_identifier": "flow_median",
            "target_datastream_id": str(forecast_datastream.uid),
        }
    ],
    interval=1,
    interval_period="days",
    enabled=True,
)

print(f"{geoglows_etl_task.name}: {geoglows_etl_task.uid}")
print(f"Enabled: {geoglows_etl_task.enabled}")
print(f"Next run: {geoglows_etl_task.next_run_at}")


Daily Nzoia GEOGLOWS Streamflow Forecast: 01a05721-cb41-74b7-913c-80e1cc24a6f5
Enabled: True
Next run: 2026-09-01 09:23:47.659334+00:00


### **Optional: Trigger the ETL Task Manually**

You can trigger the ETL Task immediately to test the configuration instead of waiting for the next scheduled run.

> This step requires HydroServer's orchestration worker to be running. It will not execute on the public Playground if orchestration is disabled.


In [17]:
# Trigger the ETL task immediately
task_run = geoglows_etl_task.trigger()

print(f"Task triggered. Run ID: {task_run.id}")

# Give the worker time to execute the task
time.sleep(10)

# Retrieve the latest task run
runs = geoglows_etl_task.list_runs()
latest_run = runs[0]

print(f"Status: {latest_run.status}")
print(f"Message: {latest_run.message}")


HTTPError: 500 Server Error: Internal Server Error for url: https://playground.hydroserver.org/api/data/etl/tasks/01a05721-cb41-74b7-913c-80e1cc24a6f5/trigger

### **Verify the Forecast Observations**

After a successful ETL Task run, retrieve the observations from the forecast Datastream to confirm that GEOGLOWS values were loaded.


In [ ]:
# Retrieve all observations currently stored in the forecast Datastream
result = forecast_datastream.get_observations(fetch_all=True)

print(f"Observations in Datastream: {len(result.dataframe)}")
display(result.dataframe.tail())


### **What Happens Automatically?**

Once the task is enabled and HydroServer orchestration is running:

**GEOGLOWS API → Data Connection → ETL Task → HydroServer Datastream**

HydroServer's orchestration worker executes the ETL Task according to the configured schedule, so the Python notebook does not need to remain open.
